In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
df = pd.read_excel('../data/online_retail_II.xlsx',
                   sheet_name='online_retail_II')

df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df.dropna(subset=['Customer ID'])
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df['Revenue'] = df['Quantity'] * df['Price']
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

print(f"Rows loaded: {len(df):,}")
print(f"Revenue: £{df['Revenue'].sum():,.0f}")

Rows loaded: 793,309
Revenue: £17,324,932


In [3]:
uk_df = df[df['Country'] == 'United Kingdom'].copy()

print(f"UK transactions: {len(uk_df):,}")
print(f"UK invoices: {uk_df['Invoice'].nunique():,}")

basket = uk_df.groupby(
    ['Invoice', 'Description'])['Quantity'].sum().unstack(fill_value=0)
basket_bool = basket.map(lambda x: True if x > 0 else False)

print(f"Basket matrix shape: {basket_bool.shape}")
print("Running Apriori... (may take 1-2 minutes)")

frequent_items = apriori(basket_bool, min_support=0.02,
                         use_colnames=True)
rules = association_rules(frequent_items, metric='lift',
                          min_threshold=1.0)
rules = rules[rules['confidence'] >= 0.5].sort_values(
    'lift', ascending=False)

rules['antecedents'] = rules['antecedents'].apply(
    lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(
    lambda x: ', '.join(list(x)))

top10 = rules[['antecedents','consequents',
               'support','confidence','lift']].head(10)
top10['support'] = top10['support'].round(3)
top10['confidence'] = top10['confidence'].round(3)
top10['lift'] = top10['lift'].round(2)

print("\n── Top 10 association rules by lift ──")
print(top10.to_string(index=False))

UK transactions: 714,245
UK invoices: 33,085
Basket matrix shape: (33085, 5247)
Running Apriori... (may take 1-2 minutes)

── Top 10 association rules by lift ──
                      antecedents                        consequents  support  confidence  lift
   SWEETHEART CERAMIC TRINKET BOX     STRAWBERRY CERAMIC TRINKET BOX    0.023       0.692 13.82
WOODEN PICTURE FRAME WHITE FINISH        WOODEN FRAME ANTIQUE WHITE     0.029       0.603 11.70
      WOODEN FRAME ANTIQUE WHITE   WOODEN PICTURE FRAME WHITE FINISH    0.029       0.565 11.70
         LOVE BUILDING BLOCK WORD           HOME BUILDING BLOCK WORD    0.023       0.530  9.97
 RED HANGING HEART T-LIGHT HOLDER WHITE HANGING HEART T-LIGHT HOLDER    0.033       0.707  5.00


In [4]:
# Rebuild RFM for export
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum')
).reset_index()
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5,
                          labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),
                          q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),
                          q=5, labels=[1,2,3,4,5]).astype(int)

def assign_segment(row):
    r, f = row['R_Score'], row['F_Score']
    if r >= 4 and f >= 4:    return 'Champions'
    elif r >= 3 and f >= 3:  return 'Loyal'
    elif r >= 4 and f <= 2:  return 'New customers'
    elif r <= 2 and f >= 3:  return 'At risk'
    elif r <= 2 and f <= 2:  return 'Lost'
    else:                    return 'Potential loyalists'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# CLV
customer_stats = df.groupby('Customer ID').agg(
    total_revenue=('Revenue', 'sum'),
    total_orders=('Invoice', 'nunique'),
    first_purchase=('InvoiceDate', 'min'),
    last_purchase=('InvoiceDate', 'max')
).reset_index()
customer_stats['lifespan_days'] = (
    customer_stats['last_purchase'] -
    customer_stats['first_purchase']).dt.days.clip(lower=1)
customer_stats['AOV'] = (
    customer_stats['total_revenue'] / customer_stats['total_orders'])
clv_df = customer_stats.merge(
    rfm[['Customer ID', 'Segment']], on='Customer ID')

# Monthly KPIs
monthly = df.groupby('YearMonth').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('Invoice', 'nunique'),
    Customers=('Customer ID', 'nunique')
).reset_index()
monthly['AOV'] = monthly['Revenue'] / monthly['Orders']
monthly['YearMonth_Date'] = monthly['YearMonth'].dt.to_timestamp() \
                                .dt.strftime('%Y-%m-%d')
monthly['YearMonth'] = monthly['YearMonth'].astype(str)
monthly.to_csv('../outputs/monthly_kpis.csv', index=False)

# RFM + CLV
clv_df.to_csv('../outputs/rfm_segments.csv', index=False)

# Cohort retention
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceDate'] \
                      .transform('min').dt.to_period('M')
df['CohortIndex'] = (
    df['YearMonth'] - df['CohortMonth']).apply(lambda x: x.n)
cohort_data = df.groupby(
    ['CohortMonth','CohortIndex'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot_table(
    index='CohortMonth', columns='CohortIndex', values='Customer ID')
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100
retention_export = retention.iloc[:, :12].copy()
retention_export.index = retention_export.index.astype(str)
retention_export.to_csv('../outputs/cohort_retention.csv')

# Country revenue
country_revenue = df.groupby('Country').agg(
    Revenue=('Revenue', 'sum'),
    Orders=('Invoice', 'nunique'),
    Customers=('Customer ID', 'nunique')
).reset_index().sort_values('Revenue', ascending=False)
country_revenue.to_csv('../outputs/country_revenue.csv', index=False)

# Association rules
rules.head(20).to_csv('../outputs/association_rules.csv', index=False)

print("── All exports complete ──")
for f in sorted(os.listdir('../outputs')):
    size = os.path.getsize(f'../outputs/{f}') / 1024
    print(f"  {f:<40} {size:.1f} KB")

── All exports complete ──
  01_revenue_trend.png                     125.9 KB
  02_rfm_segments.png                      62.7 KB
  03_clv_by_segment.png                    49.9 KB
  04_cohort_retention.png                  207.3 KB
  association_rules.csv                    1.5 KB
  cohort_retention.csv                     4.2 KB
  country_revenue.csv                      1.0 KB
  monthly_kpis.csv                         1.5 KB
  rfm_segments.csv                         464.9 KB
